# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the output file automatically.
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_uq_" + datetime_str + ".zarr"

output_file = "./results/patrolling_uq_draft2_test18.zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
from multiprocessing import Process
from args import *

MAX_PROCESSES = 200
GPUS = [0, 1, 2, 3]
GPU_AUTO_SELECT = True

def get_gpu_least_used():
    gpu_memory = os.popen("nvidia-smi --query-gpu=memory.used --format=csv").read()
    gpu_memory = gpu_memory.split("\n")[1:-1]
    gpu_memory = [int(x.split()[0]) for x in gpu_memory]
    gpu_memory = [gpu_memory[i] if i in GPUS else float('inf') for i in range(len(gpu_memory))]
    least_used_gpu = gpu_memory.index(min(gpu_memory))
    return least_used_gpu

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(0)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(3)
default_args["episode_length"] = wrap_arg_val(150)
default_args["seed"] = wrap_arg_val(2)

default_args["observation_mask"] = wrap_arg_val(True)

# default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)

# default_args["graph_name"] = wrap_arg_val("cumberland")
# default_args["graph_file"] = wrap_arg_val("../../patrolling_zoo/graphs/cumberland.graph")
default_args["graph_random"] = wrap_arg_val(True)

default_args["observation_radius_random_min"] = wrap_arg_val(0.0)
default_args["observation_radius_random_max"] = wrap_arg_val(0.0)

# Evaluation

In [3]:
import numpy as np
obs_radii = np.arange(5.0, 51.0, 10.0).tolist()
uq_noise_stds = np.arange(0.0, 10.1, 2.0).tolist()

# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    process_args = []

    def add_process(args):
        process_args.append(args)
    
    def start_process(args):
        p = Process(target=main, args=(args,))
        p.start()
        return p

    for obs_radius in obs_radii:
        for uq_noise_std in uq_noise_stds:
            # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-draft1/wandb/run-20260513_212509-jhgu4hk4/files", default_args)
            # args["eval_series"] = "Learned Comms. + Prediction"
            # args["experiment_name"] = wrap_arg_val(f"uq_noise_std_{uq_noise_std:.1f}")
            # args["eval_prediction_uq_noise"] = wrap_arg_val(uq_noise_std)
            # add_process(args)

            args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260516_113426-3dtg6gro/files", default_args)
            args["eval_series"] = f"UQ Noise Std {uq_noise_std:.1f}, Obs Radius {obs_radius:.1f}"
            args["experiment_name"] = wrap_arg_val(f"uq_noise_std_{uq_noise_std:.1f}_obs_radius_{obs_radius:.1f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            args["eval_prediction_uq_noise"] = wrap_arg_val(uq_noise_std)
            args["diffusion_autoregression_steps"] = wrap_arg_val(7)
            add_process(args)
        
        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260516_113426-3dtg6gro/files", default_args)
        args["eval_series"] = f"UQ Zeroed, Obs Radius {obs_radius:.1f}"
        args["experiment_name"] = wrap_arg_val(f"uq_zero_obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["eval_prediction_uq_multiplier"] = wrap_arg_val(0.0)
        args["diffusion_autoregression_steps"] = wrap_arg_val(7)
        add_process(args)

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(process_args) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(process_args) > 0:
            args = process_args.pop(0)

            # Get least used GPU and set it in args.
            if GPU_AUTO_SELECT:
                least_used_gpu = get_gpu_least_used()
                args["cuda_idx"] = wrap_arg_val(least_used_gpu)
                args["cuda_idx_predictor"] = wrap_arg_val(least_used_gpu)

            p = start_process(convert_args(args))
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                print(f"Process {p.pid} finished with exit code {p.exitcode}. Running: {len(running_processes)-1}, queued: {len(process_args)}, failed: {len(failed_processes)}")
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors_with_comm_uq_threshold', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--comm_uq_threshold_temperature', '0.1', '--communication_model', 'none', '--communication_probability', '0', '--critic_hidden_size', '64

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors_with_comm_uq_threshold', '--action_neighbors_max_degree', '10', '--action_velocity_acceptance_radius', '5', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'mappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--comm_uq_threshold_temperature', '0.1'

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:162: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5, 9], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '101', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


u are choosing to use mappo, we set use_recurrent_policy & use_naive_recurrent_policy to be False
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.


...

R_Critic: Use CNN: False, Use MLP: True
Creating ensemble of 3 predictors.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.


/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260516_113426-3dtg6gro/files
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260516_113426-3dtg6gro/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260516_113426-3dtg6gro/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episod

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260516_113426-3dtg6gro/files
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
QmasPolicy: Using diffusion model type gnn with prediction horizon 8 and transition dimension 7714.
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260516_113426-3dtg6gro/files
Initialized QmasAlgorithm with 3 predictors. Threaded training: True
Buffer initialized with episode length 150
Restored models from /data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260516_113426-3dtg6gro/files

# Analysis

In [4]:
import zarr
output = zarr.open(output_file, mode='r')
print(output.tree())
print(f"Output file: {os.path.abspath(output_file)}")
print(f"To load results, set in first cell: output_file = '{os.path.abspath(output_file)}'")

/
 ├── UQ Noise Std 0.0, Obs Radius 15.0
 │   └── uq_noise_std_0.0_obs_radius_15.0
 │       ├── agent_count (3, 150) float32
 │       ├── avg_idleness (3, 150) float32
 │       ├── communication
 │       │   └── requests_made (3, 150) float32
 │       ├── deltaSteps (3, 150, 4, 1) float32
 │       ├── node_visits
 │       │   ├── node_0 (3, 150) float32
 │       │   ├── node_1 (3, 150) float32
 │       │   ├── node_10 (3, 150) float32
 │       │   ├── node_11 (3, 150) float32
 │       │   ├── node_12 (3, 150) float32
 │       │   ├── node_13 (3, 150) float32
 │       │   ├── node_14 (3, 150) float32
 │       │   ├── node_15 (3, 150) float32
 │       │   ├── node_16 (3, 150) float32
 │       │   ├── node_17 (3, 150) float32
 │       │   ├── node_18 (3, 150) float32
 │       │   ├── node_19 (3, 150) float32
 │       │   ├── node_2 (3, 150) float32
 │       │   ├── node_20 (3, 150) float32
 │       │   ├── node_21 (3, 150) float32
 │       │   ├── node_22 (3, 150) float32
 │       │   ├──

In [5]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from collections import defaultdict

# --- Collect all experiments grouped by observation radius ---
# data_by_radius[obs_r] = list of (uq_noise_std, requests)
data_by_radius = defaultdict(list)
# zeroed_by_radius[obs_r] = requests (UQ multiplier = 0 condition)
zeroed_by_radius = {}

for series_key in output:
    for exp_key in output[series_key]:
        exp = output[series_key][exp_key]

        obs_r   = float(exp.attrs.get('observation_radius', 0.0))
        uq_noise = float(exp.attrs.get('eval_prediction_uq_noise', 0.0))
        uq_mult  = float(exp.attrs.get('eval_prediction_uq_multiplier', 1.0))

        requests = np.sum(exp['communication']['requests_made'][:])

        if uq_mult == 0.0:
            zeroed_by_radius[obs_r] = requests
        else:
            data_by_radius[obs_r].append((uq_noise, requests))

# --- Plot ---
fig = go.Figure()
colors = px.colors.qualitative.Plotly

for i, obs_r in enumerate(sorted(data_by_radius.keys())):
    color = colors[i % len(colors)]
    points = sorted(data_by_radius[obs_r], key=lambda p: p[0])
    x = [p[0] for p in points]
    y = [p[1] for p in points]

    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        name=f"Obs Radius {obs_r:.1f}",
        legendgroup=f"Obs Radius {obs_r:.1f}",
        line=dict(color=color),
    ))

    # Add UQ-zeroed point as a distinct marker on the same legend group
    if obs_r in zeroed_by_radius:
        fig.add_trace(go.Scatter(
            x=[0],
            y=[zeroed_by_radius[obs_r]],
            mode='markers',
            name=f"Obs Radius {obs_r:.1f} (UQ Zeroed)",
            legendgroup=f"Obs Radius {obs_r:.1f}",
            showlegend=True,
            marker=dict(color=color, symbol='x', size=10),
        ))

fig.update_layout(
    title="Communication Requests vs. Mean Uncertainty Magnitude",
    xaxis_title="UQ Noise Std (Uncertainty Magnitude)",
    yaxis_title="Total Communication Requests (per episode)",
    height=600,
    hovermode='closest',
    autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
)
fig.show()
